# Sentiment and Theme Classification of Product Reviews
## Project Live Demo Walkthrough

This notebook provides a step-by-step walkthrough of our data mining pipeline for the 5-minute class presentation. It demonstrates:
1. Loading the dataset
2. Text preprocessing and feature engineering (Character-level TF-IDF)
3. AI-Assisted labeling code and agreement metrics
4. Loading trained classical ML models and displaying evaluation tables
5. Rendering comparison visualizations and confusion matrices
6. An **interactive prediction cell** to classify custom user reviews.

### Step 1: Loading the Dataset

In [ ]:
import pandas as pd
import numpy as np
import os

# Load the labeled dataset
csv_path = "reviews_labeled_llm.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Successfully loaded dataset of {df.shape[0]} reviews.")
    print("\nSample records:")
    display(df[['reviews.text', 'sentiment', 'llm_sentiment', 'llm_category']].head(3))
else:
    print(f"Error: '{csv_path}' not found. Please verify project files.")

### Step 2: Text Preprocessing
Here we show the normalization steps (lowercasing, punctuation stripping) and how the text maps to numerical vectors using character-level n-grams.

In [ ]:
import re
import pickle

def clean_text(text):
    # Lowercase and strip punctuation
    text = str(text).lower()
    text = re.sub(r'[^\w\s\s]', '', text)  # Keep letters, numbers, and basic spacing
    return text.strip()

sample_text = "I initially had trouble deciding between the paperwhite and the voyage! It's brokene."
print("Original text:", sample_text)
print("Cleaned text: ", clean_text(sample_text))

# Load pre-fitted TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

print(f"\nTF-IDF Vectorizer details:")
print(f"  Analyzer Type:    {vectorizer.analyzer}")
print(f"  N-Gram Range:     {vectorizer.ngram_range}")
print(f"  Max Features:     {vectorizer.max_features}")

### Step 3: AI-Assisted Labeling & Agreement Rate
The code below demonstrates how we invoked Gemini (via Vertex AI) in parallel and structured the prompts to output JSON schemas.

In [ ]:
system_instruction = """
You are an expert customer feedback analyst.
Analyze the product review and output a JSON response containing:
- llm_sentiment: 'Positive', 'Neutral', or 'Negative'
- llm_category: 'Product quality issue', 'Feature request', 'Price complaint', 'Customer service issue', or 'Other'
"""
print("System Instruction Prompt:")
print(system_instruction.strip())

# Agreement metrics from Phase 2 validation
print("\n--- Human-AI Validation Results (n=1,177) ---")
print("Sentiment Agreement Accuracy: 82.24%")
print("Cohen's Kappa Score:          0.4631 (Moderate Agreement)")

### Step 4: Loading Trained Classifiers & Comparison
We load our trained Naive Bayes, Logistic Regression, and Random Forest models and display the performance tables computed in Phase 5.

In [ ]:
sentiment_metrics = pd.read_csv("sentiment_metrics_comparison.csv")
category_metrics = pd.read_csv("category_metrics_comparison.csv")

print("\n--- Task 1: Sentiment Performance (vs. Human ratings) ---")
display(sentiment_metrics[['Model', 'Accuracy', 'Macro_Precision', 'Macro_Recall', 'Macro_F1', 'Weighted_F1']])

print("\n--- Task 2: Theme/Complaint Performance (vs. LLM targets) ---")
display(category_metrics[['Model', 'Accuracy', 'Macro_Precision', 'Macro_Recall', 'Macro_F1', 'Weighted_F1']])

### Step 5: Rendering Comparison Charts and Confusion Matrices

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].imshow(mpimg.imread('sentiment_comparison.png'))
axes[0, 0].axis('off')
axes[0, 0].set_title("Sentiment Metrics Comparison")

axes[0, 1].imshow(mpimg.imread('category_comparison.png'))
axes[0, 1].axis('off')
axes[0, 1].set_title("Theme Metrics Comparison")

axes[1, 0].imshow(mpimg.imread('sentiment_confusion.png'))
axes[1, 0].axis('off')
axes[1, 0].set_title("Sentiment Confusion Matrix")

axes[1, 1].imshow(mpimg.imread('category_confusion.png'))
axes[1, 1].axis('off')
axes[1, 1].set_title("Theme Confusion Matrix")

plt.tight_layout()
plt.show()

### Step 6: Interactive Review Predictor
Type any custom customer review in the field below, run the cell, and get real-time sentiment and theme classification using our trained Random Forest models!

In [ ]:
# Load pre-trained Random Forest models
with open("sentiment_random_forest.pkl", "rb") as f:
    sentiment_model = pickle.load(f)

with open("category_random_forest.pkl", "rb") as f:
    category_model = pickle.load(f)

def classify_user_review(text_input):
    if not text_input.strip():
        print("Please enter a valid review text.")
        return
        
    # 1. Preprocess
    cleaned = clean_text(text_input)
    
    # 2. Vectorize
    features = vectorizer.transform([cleaned])
    
    # 3. Predict
    sentiment_pred = sentiment_model.predict(features)[0]
    theme_pred = category_model.predict(features)[0]
    
    print("\n=== Real-Time Classification Results ===")
    print(f"Review Text:  \"{text_input}\"")
    print(f"Sentiment:    {sentiment_pred}")
    print(f"Theme/Class:  {theme_pred}")
    print("========================================")

# Enter your custom review text here!
user_review = "The screen arrived cracked and has dead pixels. Terribly disappointed."

classify_user_review(user_review)